In [0]:
from pyspark.sql import functions as F
 
CATALOG = "workspace"
SCHEMA = "fifa_project"


### --- silver_country ---

In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.bronze_country")
silver_country = df.select(
    F.col("id").alias("country_id"),
    F.col("name").alias("country_name")
)
silver_country.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_country")
print(f"silver_country: {silver_country.count()} rows")

### --- silver_league ---


In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.bronze_league")
silver_league = df.select(
    F.col("id").alias("league_id"),
    F.col("country_id"),
    F.col("name").alias("league_name")
)
silver_league.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_league")
print(f"silver_league: {silver_league.count()} rows")

### --- silver_team ---

In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.bronze_team")
silver_team = df.select(
    F.col("team_api_id"),
    F.col("team_long_name"),
    F.col("team_short_name")
).dropDuplicates(["team_api_id"])
silver_team.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_team")
print(f"silver_team: {silver_team.count()} rows")

### --- silver_team_attributes ---

In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.bronze_team_attributes")
silver_team_attributes = (
    df.withColumn("date", F.to_date("date"))
      .select(
          "team_api_id", "date",
          "buildUpPlaySpeed", "buildUpPlayPassing",
          "chanceCreationPassing", "chanceCreationShooting",
          "defencePressure", "defenceAggression", "defenceTeamWidth"
      )
      .dropna(subset=["team_api_id", "date"])
)
silver_team_attributes.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_team_attributes")
print(f"silver_team_attributes: {silver_team_attributes.count()} rows")

### --- silver_player ---

In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.bronze_player")
silver_player = (
    df.withColumn("birthday", F.to_date("birthday"))
      .select(
          "player_api_id", "player_name", "birthday",
          F.col("height").cast("double"),
          F.col("weight").cast("double")
      )
      .dropDuplicates(["player_api_id"])
)
silver_player.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_player")
print(f"silver_player: {silver_player.count()} rows")

### --- silver_player_attributes ---

In [0]:
df = spark.table(f"{CATALOG}.{SCHEMA}.bronze_player_attributes")
silver_player_attributes = (
    df.withColumn("date", F.to_date("date"))
      .select(
          "player_api_id", "date",
          F.col("overall_rating").cast("int"),
          F.col("potential").cast("int"),
          "preferred_foot",
          F.col("crossing").cast("int"),
          F.col("finishing").cast("int"),
          F.col("dribbling").cast("int"),
          F.col("sprint_speed").cast("int"),
          F.col("stamina").cast("int")
      )
      .dropna(subset=["player_api_id", "date", "overall_rating"])
)
silver_player_attributes.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_player_attributes")
print(f"silver_player_attributes: {silver_player_attributes.count()} rows")

### --- silver_match ---


In [0]:
# Drop the 100+ betting odds columns; keep core match info + starting lineups
df = spark.table(f"{CATALOG}.{SCHEMA}.bronze_match")
 
lineup_cols = [f"home_player_{i}" for i in range(1, 12)] + [f"away_player_{i}" for i in range(1, 12)]
 
silver_match = (
    df.withColumn("date", F.to_date("date"))
      .select(
          F.col("match_api_id"),
          "country_id", "league_id", "season", "stage", "date",
          "home_team_api_id", "away_team_api_id",
          F.col("home_team_goal").cast("int"),
          F.col("away_team_goal").cast("int"),
          *lineup_cols
      )
      .dropna(subset=["match_api_id", "date", "home_team_api_id", "away_team_api_id"])
)
silver_match.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.silver_match")
print(f"silver_match: {silver_match.count()} rows")

In [0]:
# Final check
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA} LIKE 'silver_*'"))